# foretree mode test notebook

This notebook exercises `foretree` in multiple configurations:
- axis-only histogram
- k-way histogram
- oblique histogram (`ObliqueMode.Full`)
- oblique histogram auto chooser (`ObliqueMode.Auto`)
- hybrid/exact axis
- hybrid/exact oblique full
- hybrid/exact oblique interaction-seeded (`ObliqueMode.InteractionSeeded`)
- binned training/inference path
- sklearn DecisionTreeRegressor baselines (raw + binned)
- binary gradient/hessian smoke test

Run all cells top-to-bottom.

In [1]:
from pathlib import Path
import sys
import time
import numpy as np

# Prefer tree/build over root build, regardless of notebook cwd.
ROOT = Path.cwd().resolve()
cand_order = [
    ROOT / 'tree' / 'build',
    ROOT / 'build',
    ROOT.parent / 'tree' / 'build',
    ROOT.parent / 'build',
]
existing = []
for cand in cand_order:
    if cand.exists() and cand not in existing:
        existing.append(cand)
for cand in reversed(existing):
    sys.path.insert(0, str(cand))

import foretree

print('foretree:', foretree.__file__)
print('numpy:', np.__version__)


foretree: /data/solver/ad/tree/build/foretree.cpython-313-x86_64-linux-gnu.so
numpy: 2.2.6


In [2]:
def sigmoid(x):
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))


def mse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.mean((y_pred - y_true) ** 2))


def logloss(y_true, prob):
    y_true = np.asarray(y_true, dtype=np.float64)
    prob = np.clip(np.asarray(prob, dtype=np.float64), 1e-12, 1.0 - 1e-12)
    return float(-np.mean(y_true * np.log(prob) + (1.0 - y_true) * np.log(1.0 - prob)))


def accuracy(y_true, prob, thr=0.5):
    y_true = np.asarray(y_true, dtype=np.float64)
    pred = (np.asarray(prob, dtype=np.float64) >= thr).astype(np.float64)
    return float(np.mean(pred == y_true))


def grad_hess_squared(y, margin=None):
    y = np.asarray(y, dtype=np.float64)
    if margin is None:
        margin = np.zeros_like(y)
    g = margin - y
    h = np.ones_like(y)
    return g.astype(np.float64), h.astype(np.float64)


def grad_hess_logloss(y, margin=None):
    y = np.asarray(y, dtype=np.float64)
    if margin is None:
        margin = np.zeros_like(y)
    p = sigmoid(margin)
    g = p - y
    h = np.maximum(1e-12, p * (1.0 - p))
    return g.astype(np.float64), h.astype(np.float64)


def make_regression_data(n=20000, p=16, seed=7, missing_rate=0.03):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p)).astype(np.float64)

    # Two pseudo-categorical features (to stress k-way mode).
    X[:, 0] = rng.integers(0, 6, size=n)
    X[:, 1] = rng.integers(0, 4, size=n)

    y = (
        2.2 * (X[:, 0] == 3)
        - 1.6 * (X[:, 0] == 1)
        + 1.3 * (X[:, 1] == 2)
        + 0.8 * X[:, 2] * X[:, 3]
        - 1.1 * np.sin(X[:, 4])
        + 0.35 * (X[:, 5] ** 2)
        + rng.normal(0.0, 0.25, size=n)
    ).astype(np.float64)

    miss = rng.random((n, p)) < missing_rate
    X[miss] = np.nan
    return X, y


def make_binary_data(n=16000, p=12, seed=123, missing_rate=0.02):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n, p)).astype(np.float64)
    X[:, 0] = rng.integers(0, 5, size=n)
    X[:, 1] = rng.integers(0, 3, size=n)

    z = (
        1.8 * (X[:, 0] == 2)
        - 1.2 * (X[:, 0] == 4)
        + 1.0 * (X[:, 1] == 1)
        + 0.7 * X[:, 2] * X[:, 3]
        - 0.8 * np.sin(X[:, 4])
        + rng.normal(0.0, 0.35, size=n)
    )
    y = (z > 0.0).astype(np.float64)

    miss = rng.random((n, p)) < missing_rate
    X[miss] = np.nan
    return X, y


In [3]:
# Regression dataset + shared binner/codes.
X, y = make_regression_data()
n, p = X.shape
rng = np.random.default_rng(99)
perm = rng.permutation(n)
split = int(0.8 * n)
tr_idx, te_idx = perm[:split], perm[split:]

X_tr, y_tr = X[tr_idx], y[tr_idx]
X_te, y_te = X[te_idx], y[te_idx]

g_tr, h_tr = grad_hess_squared(y_tr)

hcfg = foretree.HistogramConfig()
hcfg.method = "adaptive"
hcfg.max_bins = 256
hcfg.use_missing_bin = True

ghs = foretree.GradientHistogramSystem(hcfg)
ghs.fit_bins(X_tr, g_tr, h_tr)

Xb_tr, miss_id = ghs.prebin_dataset(X_tr)
Xb_te, _ = ghs.prebin_matrix(X_te)

print("train shape:", X_tr.shape, "test shape:", X_te.shape)
print("binned train:", Xb_tr.shape, "binned test:", Xb_te.shape)
print("missing_bin_id:", miss_id)
print("bin allocation summary:")
print(ghs.get_bin_allocation_summary())

train shape: (16000, 16) test shape: (4000, 16)
binned train: (16000, 16) binned test: (4000, 16)
missing_bin_id: 139
bin allocation summary:
[(6, 'categorical_precheck'), (4, 'categorical_precheck'), (139, 'high_importance_complex'), (138, 'high_importance_complex'), (136, 'high_importance_complex'), (137, 'high_importance_complex'), (139, 'high_importance_complex'), (137, 'high_importance_complex'), (136, 'high_importance_complex'), (137, 'high_importance_complex'), (138, 'high_importance_complex'), (136, 'high_importance_complex'), (138, 'high_importance_complex'), (137, 'high_importance_complex'), (139, 'high_importance_complex'), (138, 'high_importance_complex')]


In [4]:
def build_cfg(**overrides):
    cfg = foretree.TreeConfig()
    cfg.max_depth = 6
    cfg.max_leaves = 31
    cfg.min_samples_split = 20
    cfg.min_samples_leaf = 8
    cfg.lambda_ = 1.0
    cfg.gamma_ = 0.0
    cfg.growth = foretree.Growth.LeafWise
    cfg.split_mode = foretree.SplitMode.Histogram
    cfg.exact_cutover = 2048

    cfg.enable_kway_splits = False
    cfg.kway_max_groups = 8
    cfg.enable_oblique_splits = False
    cfg.oblique_mode = foretree.ObliqueMode.Full
    cfg.oblique_k_features = 4
    cfg.oblique_ridge = 1e-3
    cfg.axis_vs_oblique_guard = 1.02

    cfg.interaction_seeded_oblique.pairs = 16
    cfg.interaction_seeded_oblique.max_top_features = 32
    cfg.interaction_seeded_oblique.max_var_candidates = 128
    cfg.interaction_seeded_oblique.first_i_cap = 16
    cfg.interaction_seeded_oblique.second_j_cap = 16
    cfg.interaction_seeded_oblique.ridge = 1e-3
    cfg.interaction_seeded_oblique.axis_guard_factor = 1.05
    cfg.interaction_seeded_oblique.use_axis_guard = True

    cfg.goss.enabled = False
    cfg.goss.top_rate = 0.2
    cfg.goss.other_rate = 0.1
    cfg.goss.min_node_size = 2000

    for k, v in overrides.items():
        if k.startswith("goss_"):
            setattr(cfg.goss, k[len("goss_"):], v)
        elif k.startswith("iseed_"):
            setattr(cfg.interaction_seeded_oblique, k[len("iseed_"):], v)
        else:
            setattr(cfg, k, v)
    return cfg


def clone_hist_cfg(base_cfg, method=None):
    out = foretree.HistogramConfig()
    out.method = base_cfg.method if method is None else method
    out.max_bins = base_cfg.max_bins
    out.use_missing_bin = base_cfg.use_missing_bin
    out.coarse_bins = base_cfg.coarse_bins
    out.density_aware = base_cfg.density_aware
    out.min_bins = base_cfg.min_bins
    out.target_bins = base_cfg.target_bins
    out.adaptive_binning = base_cfg.adaptive_binning
    out.importance_threshold = base_cfg.importance_threshold
    out.complexity_threshold = base_cfg.complexity_threshold
    out.use_feature_importance = base_cfg.use_feature_importance
    out.feature_importance_weights = list(base_cfg.feature_importance_weights)
    out.subsample_ratio = base_cfg.subsample_ratio
    out.min_sketch_size = base_cfg.min_sketch_size
    out.use_parallel = base_cfg.use_parallel
    out.max_workers = base_cfg.max_workers
    out.rng_seed = base_cfg.rng_seed
    out.eps = base_cfg.eps
    return out


_binner_cache = {}


def get_binner_bundle(method):
    if method not in _binner_cache:
        cfg_local = clone_hist_cfg(hcfg, method=method)
        ghs_local = foretree.GradientHistogramSystem(cfg_local)
        ghs_local.fit_bins(X_tr, g_tr, h_tr)
        xb_tr_local, _ = ghs_local.prebin_dataset(X_tr)
        xb_te_local, _ = ghs_local.prebin_matrix(X_te)
        _binner_cache[method] = {
            "ghs": ghs_local,
            "Xb_tr": xb_tr_local,
            "Xb_te": xb_te_local,
        }
    return _binner_cache[method]


def run_tree_case(name, cfg, use_raw_matrix=False, bundle=None):
    if bundle is None:
        bundle = get_binner_bundle("adaptive")

    ghs_local = bundle["ghs"]
    xb_tr_local = bundle["Xb_tr"]
    xb_te_local = bundle["Xb_te"]

    tree = foretree.UnifiedTree(cfg, ghs_local)

    # Required for Exact/Hybrid split search in current API.
    if use_raw_matrix:
        tree.set_raw_matrix(X_tr, None)

    t0 = time.perf_counter()
    tree.fit_binned(xb_tr_local, g_tr, h_tr)
    fit_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    if use_raw_matrix:
        pred_tr = np.asarray(tree.predict_binned(xb_tr_local, X_tr), dtype=np.float64)
        pred_te = np.asarray(tree.predict_binned(xb_te_local, X_te), dtype=np.float64)
    else:
        pred_tr = np.asarray(tree.predict_binned(xb_tr_local), dtype=np.float64)
        pred_te = np.asarray(tree.predict_binned(xb_te_local), dtype=np.float64)
    pred_s = time.perf_counter() - t1

    return {
        "name": name,
        "fit_s": fit_s,
        "pred_s": pred_s,
        "mse_tr": mse(y_tr, pred_tr),
        "mse_te": mse(y_te, pred_te),
        "nodes": int(tree.n_nodes),
        "leaves": int(tree.n_leaves),
        "depth": int(tree.depth),
    }, tree


In [9]:
scenarios = [
    ("axis_hist", dict(), "adaptive"),
    ("axis_hist_two_stage", dict(), "two_stage"),
    ("kway_hist", dict(enable_kway_splits=True, kway_max_groups=8), "adaptive"),
    ("kway_hist_two_stage", dict(enable_kway_splits=True, kway_max_groups=8), "two_stage"),

    # Histogram-mode oblique candidates.
    ("oblique_hist_full", dict(enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.Full, oblique_k_features=4), "adaptive"),
    ("oblique_hist_auto", dict(enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.Auto, oblique_k_features=4), "adaptive"),
    ("kway_oblique_hist_auto", dict(enable_kway_splits=True, enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.Auto, oblique_k_features=4), "adaptive"),

    ("goss_hist", dict(goss_enabled=True, goss_top_rate=0.2, goss_other_rate=0.1, goss_min_node_size=1000), "adaptive"),

    # Hybrid-mode axis/oblique tests.
    ("hybrid_axis", dict(split_mode=foretree.SplitMode.Hybrid, exact_cutover=1200), "adaptive"),
    ("hybrid_oblique_full", dict(split_mode=foretree.SplitMode.Hybrid, exact_cutover=1200, enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.Full, oblique_k_features=4), "adaptive"),
    ("hybrid_oblique_interaction", dict(split_mode=foretree.SplitMode.Hybrid, exact_cutover=1200, enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.InteractionSeeded, iseed_pairs=24, iseed_max_top_features=24, iseed_max_var_candidates=96, iseed_first_i_cap=12, iseed_second_j_cap=12), "adaptive"),
    ("hybrid_oblique_auto", dict(split_mode=foretree.SplitMode.Hybrid, exact_cutover=1200, enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.Auto, oblique_k_features=4, iseed_pairs=24), "adaptive"),

    # Exact-mode axis/oblique tests.
    ("exact_axis", dict(split_mode=foretree.SplitMode.Exact), "adaptive"),
    ("exact_oblique_full", dict(split_mode=foretree.SplitMode.Exact, enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.Full, oblique_k_features=4), "adaptive"),
    ("exact_oblique_interaction", dict(split_mode=foretree.SplitMode.Exact, enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.InteractionSeeded, iseed_pairs=24, iseed_max_top_features=24, iseed_max_var_candidates=96, iseed_first_i_cap=12, iseed_second_j_cap=12), "adaptive"),
    ("exact_oblique_auto", dict(split_mode=foretree.SplitMode.Exact, enable_oblique_splits=True, oblique_mode=foretree.ObliqueMode.Auto, oblique_k_features=4, iseed_pairs=24), "adaptive"),
]

results = []
models = {}

for name, overrides, bin_method in scenarios:
    cfg = build_cfg(**overrides)
    use_raw = cfg.split_mode != foretree.SplitMode.Histogram
    bundle = get_binner_bundle(bin_method)
    try:
        stats, model = run_tree_case(name, cfg, use_raw_matrix=use_raw, bundle=bundle)
        results.append(stats)
        models[name] = model
    except Exception as e:
        results.append({"name": name, "error": repr(e)})

ok = [r for r in results if "error" not in r]
err = [r for r in results if "error" in r]

ok = sorted(ok, key=lambda d: d["mse_te"])

if ok:
    print("name                           fit_s    pred_s   mse_tr    mse_te    nodes leaves depth")
    for r in ok:
        print(
            f"{r['name']:<30} {r['fit_s']:>6.3f}   {r['pred_s']:>6.3f}  "
            f"{r['mse_tr']:>7.4f}  {r['mse_te']:>7.4f}  {r['nodes']:>5d} {r['leaves']:>6d} {r['depth']:>5d}"
        )
else:
    print("No successful scenarios.")

if err:
    print("\nerrors:")
    for r in err:
        print(f"- {r['name']}: {r['error']}")


name                           fit_s    pred_s   mse_tr    mse_te    nodes leaves depth
kway_hist_two_stage             0.006    0.001   0.8966   0.9592     61     31     6
kway_hist                       0.020    0.001   0.8436   1.0125     61     31     6
kway_oblique_hist_auto          0.026    0.001   0.8436   1.0125     61     31     6
exact_axis                      0.046    0.001   1.1527   1.2328     61     31     6
exact_oblique_full              0.052    0.001   1.1506   1.2340     61     31     6
axis_hist                       0.002    0.001   1.1652   1.2444     61     31     6
axis_hist_two_stage             0.002    0.001   1.1557   1.2476     61     31     6
hybrid_axis                     0.011    0.001   1.1614   1.2489     61     31     6
exact_oblique_interaction       0.139    0.001   1.1612   1.2649     61     31     6
exact_oblique_auto              0.152    0.001   1.1606   1.2686     61     31     6
goss_hist                       0.005    0.001   1.1759   1.27

In [6]:
# sklearn baselines: compare raw vs foretree-binned features on the same split.
try:
    from sklearn.impute import SimpleImputer
    from sklearn.pipeline import make_pipeline
    from sklearn.tree import DecisionTreeRegressor
except Exception as e:
    sklearn_reg_tree = {'error': repr(e)}
    sklearn_reg_trees = []
    print('sklearn baseline unavailable:', e)
else:
    sk_params = dict(
        random_state=99,
        max_depth=6,
        max_leaf_nodes=31,
        min_samples_split=20,
        min_samples_leaf=8,
    )

    sklearn_reg_trees = []

    # 1) Raw features + median imputation.
    sk_raw_pipe = make_pipeline(
        SimpleImputer(strategy='median'),
        DecisionTreeRegressor(**sk_params),
    )

    t0 = time.perf_counter()
    sk_raw_pipe.fit(X_tr, y_tr)
    fit_raw_s = time.perf_counter() - t0

    t1 = time.perf_counter()
    pred_raw_tr = sk_raw_pipe.predict(X_tr)
    pred_raw_te = sk_raw_pipe.predict(X_te)
    pred_raw_s = time.perf_counter() - t1

    sk_raw_tree = sk_raw_pipe.named_steps['decisiontreeregressor']
    sklearn_reg_trees.append({
        'name': 'sklearn_raw',
        'fit_s': fit_raw_s,
        'pred_s': pred_raw_s,
        'mse_tr': mse(y_tr, pred_raw_tr),
        'mse_te': mse(y_te, pred_raw_te),
        'nodes': int(sk_raw_tree.tree_.node_count),
        'leaves': int(sk_raw_tree.get_n_leaves()),
        'depth': int(sk_raw_tree.get_depth()),
    })

    # 2) Foretree-binned features (uint16 codes) directly in sklearn.
    Xb_tr_np = np.asarray(Xb_tr, dtype=np.float64)
    Xb_te_np = np.asarray(Xb_te, dtype=np.float64)
    sk_bin_tree = DecisionTreeRegressor(**sk_params)

    t2 = time.perf_counter()
    sk_bin_tree.fit(Xb_tr_np, y_tr)
    fit_bin_s = time.perf_counter() - t2

    t3 = time.perf_counter()
    pred_bin_tr = sk_bin_tree.predict(Xb_tr_np)
    pred_bin_te = sk_bin_tree.predict(Xb_te_np)
    pred_bin_s = time.perf_counter() - t3

    sklearn_reg_trees.append({
        'name': 'sklearn_binned',
        'fit_s': fit_bin_s,
        'pred_s': pred_bin_s,
        'mse_tr': mse(y_tr, pred_bin_tr),
        'mse_te': mse(y_te, pred_bin_te),
        'nodes': int(sk_bin_tree.tree_.node_count),
        'leaves': int(sk_bin_tree.get_n_leaves()),
        'depth': int(sk_bin_tree.get_depth()),
    })

    # Backward-compatible alias used in some ad-hoc notebook snippets.
    sklearn_reg_tree = sklearn_reg_trees[0]

    print('name                 fit_s    pred_s   mse_tr    mse_te    nodes leaves depth')
    for r in sorted(sklearn_reg_trees, key=lambda d: d['mse_te']):
        print(
            f"{r['name']:<20} {r['fit_s']:>6.3f}   {r['pred_s']:>6.3f}  "
            f"{r['mse_tr']:>7.4f}  {r['mse_te']:>7.4f}  {r['nodes']:>5d} {r['leaves']:>6d} {r['depth']:>5d}"
        )

    if ok:
        best_ft = min(ok, key=lambda d: d['mse_te'])
        print(f"best foretree = {best_ft['name']} (mse_te={best_ft['mse_te']:.6f})")
        for r in sorted(sklearn_reg_trees, key=lambda d: d['mse_te']):
            print(f"delta best foretree vs {r['name']:<14} mse_te = {best_ft['mse_te'] - r['mse_te']:+.6f}")

    if len(sklearn_reg_trees) == 2:
        d = sklearn_reg_trees[1]['mse_te'] - sklearn_reg_trees[0]['mse_te']
        print(f"delta sklearn_binned - sklearn_raw mse_te = {d:+.6f}")


name                 fit_s    pred_s   mse_tr    mse_te    nodes leaves depth
sklearn_binned        0.035    0.001   1.1538   1.2405     61     31     6
sklearn_raw           0.095    0.002   1.1607   1.2423     61     31     6
best foretree = kway_hist_two_stage (mse_te=0.959238)
delta best foretree vs sklearn_binned mse_te = -0.281248
delta best foretree vs sklearn_raw    mse_te = -0.283060
delta sklearn_binned - sklearn_raw mse_te = -0.001812


In [7]:
# Quick comparison summary against axis_hist baseline.
ok_map = {r['name']: r for r in results if 'error' not in r}
if 'axis_hist' in ok_map:
    base = ok_map['axis_hist']
    print('baseline axis_hist mse_te =', round(base['mse_te'], 6))
    compare_names = [name for name, _, _ in scenarios if name != 'axis_hist']
    for k in compare_names:
        if k in ok_map:
            d = ok_map[k]['mse_te'] - base['mse_te']
            print(f"{k:<30} delta_mse_te = {d:+.6f}")
else:
    print('axis_hist baseline not available')


baseline axis_hist mse_te = 1.244376
axis_hist_two_stage            delta_mse_te = +0.003227
kway_hist                      delta_mse_te = -0.231882
kway_hist_two_stage            delta_mse_te = -0.285138
oblique_hist_full              delta_mse_te = +0.050487
oblique_hist_auto              delta_mse_te = +0.050487
kway_oblique_hist_auto         delta_mse_te = -0.231882
goss_hist                      delta_mse_te = +0.034900
hybrid_axis                    delta_mse_te = +0.004550
hybrid_oblique_full            delta_mse_te = +0.056059
hybrid_oblique_interaction     delta_mse_te = +0.049696
hybrid_oblique_auto            delta_mse_te = +0.047643
exact_axis                     delta_mse_te = -0.011568
exact_oblique_full             delta_mse_te = -0.010338
exact_oblique_interaction      delta_mse_te = +0.020504
exact_oblique_auto             delta_mse_te = +0.024190


In [8]:
# Binary objective smoke test (logloss gradients/hessians).
Xb_all, yb_all = make_binary_data()
nb, pb = Xb_all.shape
perm_b = np.random.default_rng(202).permutation(nb)
split_b = int(0.8 * nb)
tr_b, te_b = perm_b[:split_b], perm_b[split_b:]

Xb_tr_raw, yb_tr = Xb_all[tr_b], yb_all[tr_b]
Xb_te_raw, yb_te = Xb_all[te_b], yb_all[te_b]

gb_tr, hb_tr = grad_hess_logloss(yb_tr)

hcfg_b = foretree.HistogramConfig()
hcfg_b.method = "adaptive"
hcfg_b.max_bins = 256
ghs_b = foretree.GradientHistogramSystem(hcfg_b)
ghs_b.fit_bins(Xb_tr_raw, gb_tr, hb_tr)

Xb_b_tr, _ = ghs_b.prebin_dataset(Xb_tr_raw)
Xb_b_te, _ = ghs_b.prebin_matrix(Xb_te_raw)

cfg_b = build_cfg(enable_kway_splits=True, enable_oblique_splits=True, max_depth=5, max_leaves=31)
tree_b = foretree.UnifiedTree(cfg_b, ghs_b)
tree_b.fit_binned(Xb_b_tr, gb_tr, hb_tr)

margin_te = np.asarray(tree_b.predict_binned(Xb_b_te), dtype=np.float64)
prob_te = sigmoid(margin_te)

print('binary test logloss:', round(logloss(yb_te, prob_te), 6))
print('binary test accuracy:', round(accuracy(yb_te, prob_te), 6))
print('binary tree nodes/leaves/depth:', int(tree_b.n_nodes), int(tree_b.n_leaves), int(tree_b.depth))


binary test logloss: 0.420207
binary test accuracy: 0.828438
binary tree nodes/leaves/depth: 61 31 5
